# Day 2: 02 NAV History Cleaning
**Instructions:**
- Parse dates to datetime
- Sort by amfi_code + date
- Forward-fill missing NAV for holidays/weekends
- Remove duplicates
- Validate NAV > 0

In [ ]:
import pandas as pd
import os

raw_path = r'../../data/csv_upload/02_nav_history.csv'
processed_dir = r'../../data/processed/'

if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

df = pd.read_csv(raw_path)
print(f"Initial rows: {len(df)}")
df.head()

In [ ]:
# 1. Parse dates to datetime
df['date'] = pd.to_datetime(df['date'])

# 2. Remove duplicates
df = df.drop_duplicates(subset=['amfi_code', 'date'])

# 3. Validate NAV > 0
df = df[df['nav'] > 0]

# 4. Sort by amfi_code + date
df = df.sort_values(['amfi_code', 'date'])

print(f"Rows after basic cleaning: {len(df)}")

In [ ]:
# 5. Forward-fill missing NAV for holidays/weekends
def fill_dates(group):
    min_date = group['date'].min()
    max_date = group['date'].max()
    all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
    group = group.set_index('date').reindex(all_dates)
    group['nav'] = group['nav'].ffill()
    group['amfi_code'] = group['amfi_code'].ffill()
    group.index.name = 'date'
    return group.reset_index()

df_cleaned = df.groupby('amfi_code').apply(fill_dates).reset_index(drop=True)
print(f"Final rows after forward-fill: {len(df_cleaned)}")

In [ ]:
# Save to processed
df_cleaned.to_csv(os.path.join(processed_dir, 'nav_history_cleaned.csv'), index=False)
print("File saved to data/processed/nav_history_cleaned.csv")